In [27]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4
max_iters = 10000
learning_rate = 3e-4
eval_iters = 250

cuda


In [28]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))

vocab_size = len(chars)


In [29]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)


In [30]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split): 
    data = train_data if split =='train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y 


x, y = get_batch('train')


print('inputs: ')
print(x)
print('target: ')
print(y)


inputs: 
tensor([[57,  1, 58, 75, 58, 67,  1, 66],
        [65,  1, 56, 61, 58, 76,  1, 78],
        [57,  1, 72, 56, 54, 71, 56, 58],
        [11,  0,  0,  3, 43, 62, 71,  9]], device='cuda:0')
target: 
tensor([[ 1, 58, 75, 58, 67,  1, 66, 68],
        [ 1, 56, 61, 58, 76,  1, 78, 68],
        [ 1, 72, 56, 54, 71, 56, 58, 65],
        [ 0,  0,  3, 43, 62, 71,  9,  3]], device='cuda:0')


In [31]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([80]) target is tensor(1)
when input is tensor([80,  1]) target is tensor(1)
when input is tensor([80,  1,  1]) target is tensor(28)
when input is tensor([80,  1,  1, 28]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39]) target is tensor(42)
when input is tensor([80,  1,  1, 28, 39, 42]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42, 39]) target is tensor(44)
when input is tensor([80,  1,  1, 28, 39, 42, 39, 44]) target is tensor(32)


In [51]:
@torch.no_grad()
def estimate_loss(): 
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters): 
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [32]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size): 
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets = None): 
        logits = self.token_embedding_table(index)

        if targets is None: 
            loss = None
        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits, loss


    def generate(self, index, max_new_tokens): 

        for _ in range(max_new_tokens): 
            logits, loss = self.forward(index)

            logits = logits[: , -1 , : ]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index 


model  = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


    
    


!LanO) um;(PVWCyWZs]Rr(gCp88-:755wXf(Zeg!L5
 FMhDP3HD,Sg'03z
woYjhwv8?us5yYia ?pf_d;
q rZoOs8w5EeYCp,aKtX0y
NMF."l(!TW:06Um;hH6K1T"9.(uzsA]q67T"Ag?; ?(PR"udqqn*.)D*x1T"ktCRhj75,55hWjZnz J7!,S9ZM2Do2K4kIRry(0S.XN]Zo[.&SE0hF&MB;g1ns7(o*HV,(0UuH&h5wl
IOr?FCKnQ3E'*0o8q Glk[555*Sv&"LcmOeuJWpiMGxpOR9VSa),3wl9"(Wjhp5!pfX];9fl
)jb."ktuz(6'(7.(6KWmYt6*'[!?&(L hwv)Uv;!,cp48lE]iR[0-v8U_Vdk*'p4w?6;v[vv(*hJm7!1uz.*),-v9]Eluvz&w8wYYq NQ0h282?lKdSM&&CoUa3I[-Kbg;41N75A'Rc0BSaumwpEp7G&&abHv?Xkp0P&s Nmxyv U"tok'



In [53]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters): 
    if iter % eval_iters == 0: 
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, validation loss: {losses['val']:.3f}")
    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())
    

step: 0, train loss: 2.438, validation loss: 2.489
step: 250, train loss: 2.411, validation loss: 2.496
step: 500, train loss: 2.444, validation loss: 2.495
step: 750, train loss: 2.430, validation loss: 2.486
step: 1000, train loss: 2.430, validation loss: 2.499
step: 1250, train loss: 2.422, validation loss: 2.501
step: 1500, train loss: 2.423, validation loss: 2.478
step: 1750, train loss: 2.404, validation loss: 2.477
step: 2000, train loss: 2.419, validation loss: 2.506
step: 2250, train loss: 2.415, validation loss: 2.501
step: 2500, train loss: 2.451, validation loss: 2.465
step: 2750, train loss: 2.427, validation loss: 2.486
step: 3000, train loss: 2.429, validation loss: 2.481
step: 3250, train loss: 2.436, validation loss: 2.491
step: 3500, train loss: 2.432, validation loss: 2.487
step: 3750, train loss: 2.415, validation loss: 2.455
step: 4000, train loss: 2.420, validation loss: 2.473
step: 4250, train loss: 2.433, validation loss: 2.482
step: 4500, train loss: 2.428, val

In [54]:
context = torch.zeros((1,1), dtype=torch.long, device=device)

generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



" wof wantexthe se, bll sar beck.

" tir allone, Wil ouly  tarecro g orf ore athe. g se tocon irve s is NGGor s, tonde astoy. t b acan s."
ut  rthever ralakenges tre te am p pernth
y. ggis y d as my,
Jichack ge wexit thowa t f wantrd ode he the

" stst cowhowid bs a t wakeveere ce atced Cangeat
an'lmuid
" toocly ks grathaners b nded thareerts s OWhithelayo w a, RAl heal wembowebugafthig ke

e t tor
ce he f Fothe son aby ss,"

"Ind LEGalod thind ort?"
bed aged g bubeto ronouggha bealino s." ds le
